# SEED 三分类 EEG 情绪识别

本 notebook 展示项目的主要流程：数据检查、最终概率融合、提交文件检查。

## 1. 任务说明

任务目标是根据 EEG 片段完成三分类情绪识别。输入数据形状为 `62 x 400`，类别标签为 `0/1/2`。最终提交文件为 `submit/SEED.txt`，每行对应一个测试样本的预测标签。

In [ ]:
from pathlib import Path
import h5py
import numpy as np

cwd = Path.cwd().resolve()
candidates = [cwd, cwd.parent, cwd.parent.parent]
PACKAGE_DIR = None
for item in candidates:
    if (item / 'submit').exists() and (item / 'code').exists():
        PACKAGE_DIR = item
        break
if PACKAGE_DIR is None:
    PACKAGE_DIR = cwd
PROJECT_DIR = PACKAGE_DIR.parent
DATA_DIR = PROJECT_DIR / 'data_seed_link'
print('package dir:', PACKAGE_DIR)
print('project dir:', PROJECT_DIR)
print('data dir:', DATA_DIR)

## 2. 数据格式检查

In [ ]:
def read_h5(file_name):
    path = DATA_DIR / file_name
    with h5py.File(path, 'r') as f:
        X = f['X'][:]
        y = f['y'][:] if 'y' in f else None
    return X, y

def show_info(name, X, y=None):
    flat = X.reshape(X.shape[0], -1)
    print('\n' + name)
    print('shape:', X.shape)
    print('dtype:', X.dtype)
    print('nan:', int(np.isnan(X).sum()), 'inf:', int(np.isinf(X).sum()))
    print('global mean:', float(X.mean()))
    print('global std:', float(X.std()))
    print('sample std median:', float(np.median(flat.std(axis=1))))
    if y is not None:
        labels, counts = np.unique(y, return_counts=True)
        print('label counts:', dict(zip(labels.tolist(), counts.tolist())))

X_train, y_train = read_h5('train.h5')
X_val, y_val = read_h5('val.h5')
X_test, _ = read_h5('test_x_only.h5')

show_info('train', X_train, y_train)
show_info('validation', X_val, y_val)
show_info('test_x_only', X_test)

## 3. 最终概率融合

最终结果由四个候选模型的预测概率加权平均得到。该步骤只读取已经保存的概率文件，不读取测试集标签。

In [ ]:
PROB_DIR = PACKAGE_DIR / 'probs'
SUBMIT_DIR = PACKAGE_DIR / 'submit'

weights = np.array([
    0.5221866239430111,
    0.12563498149116575,
    0.2907072375990765,
    0.06147115696674659,
])

test_probs = []
for i in range(4):
    test_probs.append(np.load(PROB_DIR / 'test' / f'model{i + 1}.npy'))

final_prob = np.zeros_like(test_probs[0], dtype=float)
for prob, weight in zip(test_probs, weights):
    final_prob += prob * weight
final_prob = final_prob / final_prob.sum(axis=1, keepdims=True)

pred = final_prob.argmax(axis=1)
labels, counts = np.unique(pred, return_counts=True)
dict(zip(labels.tolist(), counts.tolist()))

In [ ]:
SUBMIT_DIR.mkdir(parents=True, exist_ok=True)
out_path = SUBMIT_DIR / 'SEED.txt'
with open(out_path, 'w', encoding='utf-8') as f:
    for label in pred:
        f.write(str(int(label)) + '\n')

print('saved:', out_path)
print('rows:', len(pred))

## 4. 提交文件检查

In [ ]:
def read_labels(path):
    labels = []
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line:
                labels.append(int(line))
    return labels

labels = read_labels(SUBMIT_DIR / 'SEED.txt')
print('rows:', len(labels))
print('valid labels:', all(x in [0, 1, 2] for x in labels))
unique, counts = np.unique(labels, return_counts=True)
print('label counts:', dict(zip(unique.tolist(), counts.tolist())))

## 5. 结果说明

最终提交文件为 `submit/SEED.txt`。该文件共 450 行，标签范围为 `0/1/2`。测试集 `X` 的基本统计量与训练集、验证集接近，因此没有发现明显的输入分布异常。最终模型仍包含概率融合和块平滑候选，因此存在一定验证集选择风险，保守备选文件为 `submit/SEED_safe.txt`。